# RAG Pipeline with Query Expansion
This notebook implements a Retrieval-Augmented Generation (RAG) pipeline. It uses **ChromaDB** for vector storage, **LangChain** for document splitting, and **Groq** for LLM generation. 
We enhance retrieval performance by using **Query Expansion** to generate multiple variations of the user's query.

## Imports

In [1]:
import os
import chromadb
from dotenv import load_dotenv
from groq import Groq
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

c:\Users\Mohammad Anas\Desktop\Training\AG0860_2_Mohammad Anas\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Environment Setup & Initialization

In [2]:
# Load environment variables
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing! Please check your .env file.")

# Initialize Groq Client
client = Groq(api_key=api_key)
model_name = 'openai/gpt-oss-120b' # Note: Make sure you use a valid Groq model name here

# Initialize Embedding Model
print("Loading Embedding Model...")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
print("Model loaded successfully!")

Loading Embedding Model...


C:\Users\Mohammad Anas\AppData\Local\Temp\ipykernel_17284\2943726564.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


Model loaded successfully!


## Database Setup Function

In [3]:
def setup_database():
    chromadb_client = chromadb.PersistentClient(path="./assignment2_vector_db")
    vectorstore = Chroma(
        collection_name='tesla_main_collection',
        embedding_function=embedding_model,
        client=chromadb_client
    )
    
    # Check if database is already created to save time on rerun
    if vectorstore._collection.count() == 0:
        print("Database is empty. Loading PDFs from directory...")
        loader = PyPDFDirectoryLoader(".") 
        documents = loader.load()
        
        if not documents:
            print("No PDFs found in the current directory!")
            return None
            
        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name='cl100k_base', chunk_size=512, chunk_overlap=16
        )
        chunks = text_splitter.split_documents(documents)
        print(f"Total {len(chunks)} chunks created from PDF. Saving to database...")
        
        vectorstore.add_documents(documents=chunks)
        print("Database creation complete.")
    else:
        print(f"Loaded existing database with {vectorstore._collection.count()} chunks.")
        
    return vectorstore.as_retriever(search_kwargs={'k': 3})

## Query Expansion & Generation Function

In [4]:
def run_query_expansion(retriever, test_query):
    print(f"Original Query: {test_query}")
    
    # 1. Query Expansion Prompt
    prompt = f"""You are a financial domain expert. Expand the following question into 3 different versions using synonyms to help with database retrieval. 
    Return ONLY a list of questions, each on a new line. Do not number them or use bullet points.
    Question: {test_query}"""
    
    try:
        response = client.chat.completions.create(
            model=model_name, 
            messages=[{'role': 'user', 'content': prompt}], 
            temperature=0
        )
        expanded_queries = response.choices[0].message.content.strip().split("\n")
    except Exception as e:
        print(f"Error during query expansion: {e}")
        expanded_queries = [test_query]

    print("\nExpanded Queries:")
    for q in expanded_queries:
        print(f" - {q}")

    # 2. Retrieval using all queries
    all_retrieved_docs = []
    for q in expanded_queries:
        if q.strip():
            docs = retriever.invoke(q)
            all_retrieved_docs.extend(docs)

    # 3. Deduplicate documents based on content
    unique_docs = {doc.page_content: doc for doc in all_retrieved_docs}.values()
    context_for_query = "\n---\n".join([doc.page_content for doc in unique_docs])

    # 4. Final Answer Generation
    final_prompt = [
        {'role': 'system', 'content': "Answer user queries ONLY using the context provided. If not found, say 'I don't know'."},
        {'role': 'user', 'content': f"<Context>\n{context_for_query}\n</Context>\n\n<Question>\n{test_query}\n</Question>"}
    ]

    print("\nGenerating Final Answer...\n")
    ans_response = client.chat.completions.create(model=model_name, messages=final_prompt, temperature=0)
    print("================ FINAL ANSWER ================\n")
    print(ans_response.choices[0].message.content.strip())

## Execution

In [5]:
# 1. Setup the Retriever (Make sure your PDFs are in the same folder)
retriever = setup_database()

# 2. Run the Query
if retriever:
    test_question = "What were the major risks related to COVID-19 for Tesla?"
    run_query_expansion(retriever, test_question)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Loaded existing database with 3337 chunks.
Original Query: What were the major risks related to COVID-19 for Tesla?

Expanded Queries:
 - What were the primary hazards associated with COVID-19 for Tesla?  
 - What were the significant threats stemming from COVID-19 that Tesla faced?  
 - What key challenges did Tesla encounter due to the COVID-19 pandemic?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Generating Final Answer...

================ FINAL ANSWER ================

**Major COVID‑19‑related risks for Tesla (as described in the filing)**  

1. **Temporary shutdowns of manufacturing facilities** – Tesla had to suspend operations at each of its worldwide factories for part of the first half of 2020.  

2. **Supply‑chain disruptions** – Key suppliers and partners (e.g., Panasonic, which makes battery cells at the Gigafactory Nevada) also faced temporary suspensions, leading to delays in receiving critical parts. The pandemic‑driven surge in demand for personal electronics created a short‑fall of microchips, and broader issues such as port congestion and intermittent supplier shutdowns added further risk.  

3. **Employee‑related actions** – Tesla instituted temporary employee furloughs and compensation reductions while U.S. operations were scaled back.  

4. **Impediments to vehicle‑delivery and energy‑product deployment** – Reduced operations or closures at motor‑vehicle dep

 ## Done By Mohammad Anas (AG0860)